In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ETAPA 0 — Conversão Excel para CSV

In [2]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_origem = "Relatório de Fechamento.xlsx"

df = pd.read_excel(arquivo_origem)


ETAPA 1 — Filtro Fluxo de Caixa

In [3]:
df.columns = df.columns.str.strip()

df = df[df["Objeto"].str.contains("fluxo de caixa", case=False, na=False)]

df.to_excel("relatorio.xlsx", index=False)

print(f"Total de linhas após filtro: {len(df)}")

Total de linhas após filtro: 32943


1.1 Remoção de duplicadas

In [4]:
import pandas as pd

arquivo = "relatorio.xlsx"

col_pasta = "Pasta"
col_data = "Data Cálculo"

df = pd.read_excel(arquivo)

linhas_iniciais = len(df)

# ======================================
# Converter coluna de data corretamente
# ======================================

if pd.api.types.is_numeric_dtype(df[col_data]):
    
    # Caso seja número Excel
    df[col_data] = pd.to_datetime(
        df[col_data],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )

else:
    
    # Caso seja texto ou datetime
    df[col_data] = pd.to_datetime(
        df[col_data],
        errors="coerce",
        dayfirst=True
    )

# ======================================
# Ordenar pela data mais recente
# ======================================

df = df.sort_values(by=col_data, ascending=False)

# ======================================
# Remover duplicados
# ======================================

df = df.drop_duplicates(subset=[col_pasta], keep="first")

linhas_finais = len(df)

# ======================================
# Salvar base limpa
# ======================================

df.to_excel(arquivo, index=False)

print("Limpeza de duplicados concluída.")
print(f"Linhas antes: {linhas_iniciais}")
print(f"Linhas depois: {linhas_finais}")
print(f"Duplicados removidos: {linhas_iniciais - linhas_finais}")

Limpeza de duplicados concluída.
Linhas antes: 32943
Linhas depois: 5910
Duplicados removidos: 27033


ETAPA 2 — Exclusão por Categorias

In [5]:
# ETAPA 2 — Excluídos

df = pd.read_excel("relatorio.xlsx")

df.columns = df.columns.str.strip()

mask = (
    df["Natureza"].isin([
        "Administrativo - Gerencial",
        "Administrativo - Protestos",
        "Criminal"
    ]) |
    (df["Natureza Financeira"].astype(str).str.upper() == "NEUTRO") |
    (df["Tipo de Ação"].astype(str).str.upper() == "NOTIFICAÇÃO EXTRAJUDICIAL") |
    (df["Assunto Principal"].astype(str).str.upper() == "RESCISÃO CONTRATUAL - LOCAÇÃO")
)

excluidos = df[mask].copy()
df = df[~mask].copy()

excluidos.to_excel("Excluidos.xlsx", index=False)
df.to_excel("relatorio.xlsx", index=False)

print("Excluídos:", len(excluidos))

Excluídos: 468


ETAPA 3 — Tratamento (AY / AZ)

In [6]:
df = pd.read_excel("relatorio.xlsx")

total_inicial = len(df)

col_AY = "Valor Pedido Objeto Corrigido"
col_AZ = "Valor Pedido"

df[col_AY] = (
    df[col_AY]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df[col_AZ] = (
    df[col_AZ]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df[col_AY] = pd.to_numeric(df[col_AY], errors="coerce")
df[col_AZ] = pd.to_numeric(df[col_AZ], errors="coerce")

mask_remove = (
    ((df[col_AY].round(2) == 0.01) & (df[col_AZ].round(2) == 0.01)) |
    ((df[col_AY].round(2) == 0.22) & (df[col_AZ].round(2) == 0.22))
)

mask_alerta = (
    (df[col_AY].round(2).isin([0.01, 0.22])) ^
    (df[col_AZ].round(2).isin([0.01, 0.22]))
)

alertas = df[mask_alerta].copy()
alertas.to_excel("alertas_valor.xlsx", index=False) #avisa se um for significativo e outro não

df = df[~(mask_remove | mask_alerta)]

total_final = len(df)

df.to_excel("relatorio.xlsx", index=False)

print("ETAPA 3 concluída ✔")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas enviadas para alertas: {len(alertas)}")
print(f"Total final após tratamento: {total_final}")
print(f"Total removido da base principal: {total_inicial - total_final}")

ETAPA 3 concluída ✔
Total inicial de linhas: 5442
Linhas enviadas para alertas: 0
Total final após tratamento: 5442
Total removido da base principal: 0


ETAPA 4 — Tratamento dos Excluídos

In [7]:
df = pd.read_excel("relatorio.xlsx")
exc = pd.read_excel("Excluidos.xlsx")

col_valor = "Valor Pedido Objeto Corrigido"

exc[col_valor] = (
    exc[col_valor]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

exc[col_valor] = pd.to_numeric(exc[col_valor], errors="coerce")

# remover definitivamente valores ≤ 1
exc_validos = exc[exc[col_valor] > 1].copy()

# reintegrar válidos
df = pd.concat([df, exc_validos], ignore_index=True)



df.to_excel("relatorio_tratado.xlsx", index=False)

print("Processos reintegrados:", len(exc_validos))
print("Total final do relatório:", len(df))

Processos reintegrados: 316
Total final do relatório: 5758


ETAPA 5 — Corte de Data

In [8]:
df = pd.read_excel("relatorio_tratado.xlsx")

total_inicial = len(df)

col_data = "Data de cadastro"

def conv_data(col):

    # tenta converter datas normais
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    # tenta converter datas Excel
    col_numeric = pd.to_numeric(col, errors="coerce")

    mask_excel = d.isna() & col_numeric.notna()

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    return d

df[col_data] = conv_data(df[col_data])

# ============================================
# >>> ALTERAR AQUI <<<
data_corte_manual = "26/02/2026"
# ============================================

data_corte = pd.to_datetime(data_corte_manual, dayfirst=True)

mask = df[col_data] >= data_corte

linhas_removidas = mask.sum()

df = df[~mask].copy()

total_final = len(df)

df.to_excel("relatorio_tratado.xlsx", index=False)

print("ETAPA 5 concluída ✔")
print(f"Data de corte aplicada: {data_corte.date()}")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas removidas (>= data de corte): {linhas_removidas}")
print(f"Total final do documento: {total_final}")

ETAPA 5 concluída ✔
Data de corte aplicada: 2026-02-26
Total inicial de linhas: 5758
Linhas removidas (>= data de corte): 0
Total final do documento: 5758


ETAPA 6 — Separar por Status

In [9]:
df = pd.read_excel("relatorio_tratado.xlsx")

total_inicial = len(df)

col_status = "Status"

# normalizar texto
df[col_status] = df[col_status].astype(str).str.strip().str.upper()

df_ativos = df[df[col_status] == "ATIVOS"]
df_baixa = df[df[col_status] == "BAIXA PROVISÓRIA"]
df_encerrados = df[df[col_status] == "ENCERRADOS"]

df_outros = df[~df[col_status].isin([
    "ATIVOS",
    "BAIXA PROVISÓRIA",
    "ENCERRADOS"
])]

df_ativos.to_excel("ATIVOS.xlsx", index=False)
df_baixa.to_excel("BAIXA_PROVISORIA.xlsx", index=False)
df_encerrados.to_excel("ENCERRADOS.xlsx", index=False)

print("ETAPA 6 concluída ✔")
print(f"Total de linhas na base: {total_inicial}")
print(f"ATIVOS: {len(df_ativos)}")
print(f"BAIXA PROVISÓRIA: {len(df_baixa)}")
print(f"ENCERRADOS: {len(df_encerrados)}")
print(f"OUTROS STATUS: {len(df_outros)}")

ETAPA 6 concluída ✔
Total de linhas na base: 5758
ATIVOS: 5203
BAIXA PROVISÓRIA: 224
ENCERRADOS: 331
OUTROS STATUS: 0


Etapa 6.1 - PRÉ CLASSIFICAÇÃO - MUDANÇA DE PROGNÓSTICO (AY/AZ)

In [10]:
import pandas as pd

df_ativos = pd.read_excel("ATIVOS.xlsx")

col_valor_atual = "Valor Pedido Objeto Corrigido"
col_valor_bp = "Valor Pedido"

for col in [col_valor_atual, col_valor_bp]:

    df_ativos[col] = (
        df_ativos[col]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df_ativos[col] = pd.to_numeric(df_ativos[col], errors="coerce")

condicao_mudanca = (
    ((df_ativos[col_valor_bp] <= 1) & (df_ativos[col_valor_atual] > 1)) |
    ((df_ativos[col_valor_atual] <= 1) & (df_ativos[col_valor_bp] > 1))
)

mudanca_prognostico = df_ativos[condicao_mudanca].copy()
ativos_normais = df_ativos[~condicao_mudanca].copy()

mudanca_prognostico.to_excel("MUDANCA_PROGNOSTICO.xlsx", index=False)
ativos_normais.to_excel("ATIVOS_NORMAIS.xlsx", index=False)

print("Mudança de Prognóstico inicial:", len(mudanca_prognostico))
print("Ativos Normais:", len(ativos_normais))

Mudança de Prognóstico inicial: 235
Ativos Normais: 4968


ETAPA 7 — Criação do SETTLED

In [11]:
# ETAPA 7 — Criação do SETTLED

baixa = pd.read_excel("BAIXA_PROVISORIA.xlsx")
enc = pd.read_excel("ENCERRADOS.xlsx")

# colunas usadas
col_data = "Data"
col_motivo_bp = "Motivo da Baixa Provisória"
col_data_enc = "Data de Encerramento"
col_motivo_enc = "Motivo Encerramento"

def conv_data(col):
    # 1️⃣ tenta converter normalmente
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")

    # 2️⃣ trata Excel (dias desde 1899)
    mask_excel = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 0) &
        (col_numeric < 60000)
    )

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    # 3️⃣ trata timestamp em nanossegundos
    mask_ns = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 1e12)
    )

    d.loc[mask_ns] = pd.to_datetime(
        col_numeric[mask_ns],
        unit="ns"
    )

    return d

baixa[col_data] = conv_data(baixa[col_data])
enc[col_data] = conv_data(enc[col_data])

print("BAIXA - datas válidas:", baixa[col_data].notna().sum())
print("BAIXA - mínima:", baixa[col_data].min())
print("BAIXA - máxima:", baixa[col_data].max())

print("ENCERRADOS - datas válidas:", enc[col_data].notna().sum())
print("ENCERRADOS - mínima:", enc[col_data].min())
print("ENCERRADOS - máxima:", enc[col_data].max())

# >>> AQUI VOCÊ ALTERA <<<
data_inicio_manual = "26/01/2026"
data_fim_manual = "25/02/2026"

ini = pd.to_datetime(data_inicio_manual, dayfirst=True)
fim = pd.to_datetime(data_fim_manual, dayfirst=True)

baixa_filtrado = baixa[(baixa[col_data] >= ini) & (baixa[col_data] <= fim)].copy()
enc_filtrado = enc[(enc[col_data] >= ini) & (enc[col_data] <= fim)].copy()

# transformação SOMENTE BAIXA PROVISÓRIA
baixa_filtrado[col_data_enc] = baixa_filtrado[col_data]
baixa_filtrado[col_motivo_enc] = baixa_filtrado[col_motivo_bp]

settled = pd.concat([baixa_filtrado, enc_filtrado], ignore_index=True)
settled.to_excel("SETTLED_2.xlsx", index=False)

print("\nETAPA 7 concluída ✔")
print(f"Período aplicado: {ini.date()} até {fim.date()}")
print(f"BAIXA PROVISÓRIA no período: {len(baixa_filtrado)}")
print(f"ENCERRADOS no período: {len(enc_filtrado)}")
print(f"Total no SETTLED: {len(settled)}")

BAIXA - datas válidas: 224
BAIXA - mínima: 2023-06-14 00:00:00
BAIXA - máxima: 2026-03-03 00:00:00
ENCERRADOS - datas válidas: 126
ENCERRADOS - mínima: 2021-01-01 00:00:00
ENCERRADOS - máxima: 2026-02-26 00:00:00

ETAPA 7 concluída ✔
Período aplicado: 2026-01-26 até 2026-02-25
BAIXA PROVISÓRIA no período: 29
ENCERRADOS no período: 4
Total no SETTLED: 33


Etapa 8.0 - Atualização do entradas pós BP

In [12]:
import pandas as pd

arquivo_excel = "Entradas_Pos_BP.xlsx"

df_pos_bp = pd.read_excel(arquivo_excel)

In [13]:
# ETAPA 8.0 — Sincronizar Entradas_Pos_BP com relatorio_tratado

import os
import pandas as pd

# =========================
# função para ler Entradas_Pos_BP com header automático
# =========================
def ler_entradas_seguro(caminho):

    df_test = pd.read_excel(caminho, header=None)

    linha_header = None

    for i in range(min(5, len(df_test))):
        linha = df_test.iloc[i].astype(str).str.strip().tolist()

        if "Pasta" in linha and "Status" in linha:
            linha_header = i
            break

    if linha_header is None:
        raise ValueError("Header não encontrado em Entradas_Pos_BP.xlsx")

    df = pd.read_excel(caminho, header=linha_header, dtype=str)

    return df


# =========================
# Ler arquivos
# =========================

if not os.path.exists("Entradas_Pos_BP.xlsx"):
    print("Arquivo Entradas_Pos_BP.xlsx não encontrado.")

elif not os.path.exists("relatorio_tratado.xlsx"):
    print("Arquivo relatorio_tratado.xlsx não encontrado.")

else:

    entradas = ler_entradas_seguro("Entradas_Pos_BP.xlsx")
    relatorio = pd.read_excel("relatorio_tratado.xlsx", dtype=str)

    # =========================
    # limpar nomes de colunas
    # =========================
    entradas.columns = entradas.columns.astype(str).str.strip()
    relatorio.columns = relatorio.columns.astype(str).str.strip()

    # =========================
    # remover colunas vazias criadas pelo Excel
    # =========================
    entradas = entradas.loc[:, ~entradas.columns.str.contains("^Unnamed", case=False, na=False)]
    relatorio = relatorio.loc[:, ~relatorio.columns.str.contains("^Unnamed", case=False, na=False)]

    col_pasta = "Pasta"
    col_status = "Status"

    # =========================
    # verificar colunas
    # =========================
    if col_pasta not in entradas.columns or col_pasta not in relatorio.columns:

        print("Coluna 'Pasta' não encontrada.")
        print("Colunas Entradas:", entradas.columns.tolist())
        print("Colunas Relatorio:", relatorio.columns.tolist())

    elif col_status not in entradas.columns or col_status not in relatorio.columns:

        print("Coluna 'Status' não encontrada.")
        print("Colunas Entradas:", entradas.columns.tolist())
        print("Colunas Relatorio:", relatorio.columns.tolist())

    else:

        # =========================
        # limpar campo Pasta
        # =========================
        entradas[col_pasta] = entradas[col_pasta].astype(str).str.strip()
        relatorio[col_pasta] = relatorio[col_pasta].astype(str).str.strip()

        entradas[col_status] = entradas[col_status].astype(str).str.strip()
        relatorio[col_status] = relatorio[col_status].astype(str).str.strip()

        # =========================
        # remover duplicatas no relatório
        # =========================
        relatorio = relatorio.drop_duplicates(subset=[col_pasta], keep="last")

        # =========================
        # criar mapa Pasta -> Status
        # =========================
        status_map = relatorio.set_index(col_pasta)[col_status]

        # =========================
        # obter status do relatório
        # =========================
        entradas["Status_relatorio"] = entradas[col_pasta].map(status_map)

        # =========================
        # identificar diferenças
        # =========================
        mask = (
            entradas["Status_relatorio"].notna() &
            (entradas[col_status] != entradas["Status_relatorio"])
        )

        total_atualizadas = mask.sum()

        # =========================
        # atualizar status
        # =========================
        entradas.loc[mask, col_status] = entradas.loc[mask, "Status_relatorio"]

        # remover coluna auxiliar
        entradas.drop(columns=["Status_relatorio"], inplace=True)

        # =========================
        # salvar
        # =========================
        entradas.to_excel("Entradas_Pos_BP.xlsx", index=False)

        print("ETAPA 8.0 concluída ✔")
        print("Status atualizados:", total_atualizadas)

ETAPA 8.0 concluída ✔
Status atualizados: 0


 ETAPA 8.1 — Entradas

In [14]:
ativos = pd.read_excel("ATIVOS_NORMAIS.xlsx")

# coluna usada
col_data_cadastro = "Data de cadastro"

def conv_data(col):
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")
    mask_excel = d.isna() & col_numeric.notna()

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    return d

ativos[col_data_cadastro] = conv_data(ativos[col_data_cadastro])

# >>> AQUI VOCÊ ALTERA <<<
limite = pd.Timestamp(2025, 8, 26)

entradas = ativos[ativos[col_data_cadastro] >= limite].copy()
ativos_restantes = ativos[ativos[col_data_cadastro] < limite].copy()

entradas.to_excel("ENTRADAS_ANALISE.xlsx", index=False)
ativos_restantes.to_excel("ATIVOS_NORMAIS.xlsx", index=False)

print("ETAPA 8 concluída ✔")
print("Entradas para análise:", len(entradas))
print("Ativos restantes:", len(ativos_restantes))

ETAPA 8 concluída ✔
Entradas para análise: 268
Ativos restantes: 4700


ETAPA 9 — Comparação com Entradas_Pos_BP

In [15]:
# ETAPA 9 — Atualizar Entradas_Pos_BP com ENTRADAS_ANALISE

import os
import pandas as pd
from pandas.errors import EmptyDataError

col_pasta = "Pasta"

# ===============================
# Ler ENTRADAS_ANALISE
# ===============================
if not os.path.exists("ENTRADAS_ANALISE.xlsx"):
    print("ENTRADAS_ANALISE.xlsx não encontrado.")
else:
    analise = pd.read_excel("ENTRADAS_ANALISE.xlsx", dtype=str)

    if analise.empty:
        print("ENTRADAS_ANALISE vazio. ETAPA 9 ignorada.")
    else:
        analise.columns = analise.columns.str.strip()
        analise[col_pasta] = analise[col_pasta].astype(str).str.strip()

        # ===============================
        # Ler Entradas_Pos_BP
        # ===============================
        if os.path.exists("Entradas_Pos_BP.xlsx"):
            try:
                bp = pd.read_excel("Entradas_Pos_BP.xlsx", dtype=str)
            except EmptyDataError:
                bp = pd.DataFrame(columns=analise.columns)
        else:
            bp = pd.DataFrame(columns=analise.columns)

        if not bp.empty:
            bp.columns = bp.columns.str.strip()
            bp[col_pasta] = bp[col_pasta].astype(str).str.strip()

        # ===============================
        # Alinhar colunas
        # ===============================
        todas_colunas = sorted(set(bp.columns).union(set(analise.columns)))

        bp = bp.reindex(columns=todas_colunas)
        analise = analise.reindex(columns=todas_colunas)

        # ===============================
        # Remover pastas antigas
        # ===============================
        bp = bp[~bp[col_pasta].isin(analise[col_pasta])]

        # ===============================
        # Concatenar
        # ===============================
        bp_final = pd.concat([bp, analise], ignore_index=True)

        # Remover duplicados finais
        bp_final = bp_final.drop_duplicates(subset=[col_pasta], keep="last")

        # ===============================
        # Salvar
        # ===============================
        bp_final.to_excel("Entradas_Pos_BP.xlsx", index=False)

        print("ETAPA 9 concluída ✔")
        print("Linhas adicionadas/substituídas:", len(analise))
        print("Total final:", len(bp_final))

ETAPA 9 concluída ✔
Linhas adicionadas/substituídas: 268
Total final: 327


 ETAPA 10 — Análise Financeira/Ativos

In [16]:
# ETAPA 10 — Análise Financeira/Ativos

import pandas as pd
import os

df = pd.read_excel("ENTRADAS_ANALISE.xlsx")

total_inicial = len(df)

col_valor_atual = "Valor Pedido Objeto Corrigido"
col_valor_bp = "Valor Pedido"
col_data_cadastro = "Data de cadastro"

df[col_valor_atual] = pd.to_numeric(
    df[col_valor_atual].astype(str).str.replace(",", "."),
    errors="coerce"
)

df[col_valor_bp] = pd.to_numeric(
    df[col_valor_bp].astype(str).str.replace(",", "."),
    errors="coerce"
)

df["comparacao"] = df[col_valor_atual] - df[col_valor_bp]

sem_var = df[df["comparacao"] == 0].copy()
com_var = df[df["comparacao"] != 0].copy()

com_var["porcentagem"] = (com_var[col_valor_atual] / com_var[col_valor_bp]) - 1

def conv_data(col):
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")
    mask_excel = d.isna() & col_numeric.notna()

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    return d

com_var[col_data_cadastro] = conv_data(com_var[col_data_cadastro])

print("Datas válidas para análise:", com_var[col_data_cadastro].notna().sum())
print("Data mínima encontrada:", com_var[col_data_cadastro].min())
print("Data máxima encontrada:", com_var[col_data_cadastro].max())

# >>> AQUI VOCÊ ALTERA <<<
ano_referencia = 2026
mes_referencia = 2

com_var["meses"] = (
    (com_var[col_data_cadastro].dt.year - ano_referencia) * 12 +
    (com_var[col_data_cadastro].dt.month - mes_referencia)
).clip(lower=1)

com_var["analise"] = com_var["porcentagem"] / com_var["meses"]

ativos_com_variacao = com_var[com_var["analise"] > 0.05].copy()
prognostico_novo = com_var[com_var["analise"] <= 0.05].copy()

ativos_com_variacao.to_excel("ATIVOS_COM_VARIACAO.xlsx", index=False)
sem_var.to_excel("ATIVOS_SEM_VARIACAO.xlsx", index=False)

if os.path.exists("MUDANCA_PROGNOSTICO.xlsx"):
    mudanca_existente = pd.read_excel("MUDANCA_PROGNOSTICO.xlsx")
    mudanca_final = pd.concat(
        [mudanca_existente, prognostico_novo],
        ignore_index=True
    )
else:
    mudanca_final = prognostico_novo

mudanca_final.to_excel("MUDANCA_PROGNOSTICO.xlsx", index=False)

print("\nETAPA 10 concluída ✔")
print(f"Ano referência: {ano_referencia}")
print(f"Mês referência: {mes_referencia}")
print(f"Total inicial: {total_inicial}")
print(f"Sem variação: {len(sem_var)}")
print(f"Com variação: {len(com_var)}")
print(f"Ativos com variação relevante: {len(ativos_com_variacao)}")
print(f"Mudança de prognóstico (novos): {len(prognostico_novo)}")
print(f"Total final em MUDANCA_PROGNOSTICO: {len(mudanca_final)}")

Datas válidas para análise: 267
Data mínima encontrada: 2025-08-27 00:00:00
Data máxima encontrada: 2026-02-18 00:00:00


C:\Users\camil\AppData\Local\Temp\ipykernel_1996\4265726391.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  mudanca_final = pd.concat(



ETAPA 10 concluída ✔
Ano referência: 2026
Mês referência: 2
Total inicial: 268
Sem variação: 1
Com variação: 267
Ativos com variação relevante: 2
Mudança de prognóstico (novos): 265
Total final em MUDANCA_PROGNOSTICO: 500


 ETAPA 11 — Atualização do Histórico

In [17]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

col_pasta = "Pasta"
col_status = "Status"
col_data_cadastro = "Data de cadastro"

# =====================================================
# Ler SETTLED_2.xlsx (novo mês)
# =====================================================

if not os.path.exists("SETTLED_2.xlsx"):
    print("SETTLED_2.xlsx não existe. ETAPA 11 ignorada.")

else:

    try:
        novo = pd.read_excel("SETTLED_2.xlsx", dtype=str)
    except EmptyDataError:
        print("SETTLED_2.xlsx vazio. ETAPA 11 ignorada.")
        novo = pd.DataFrame()

    if novo.empty:
        print("Sem dados em SETTLED_2.")
    else:

        novo.columns = novo.columns.astype(str).str.strip()

        # =====================================================
        # Ler histórico
        # =====================================================

        hist = pd.DataFrame()

        if os.path.exists("SETTLED.xlsx"):

            try:
                hist = pd.read_excel("SETTLED.xlsx", dtype=str, engine="openpyxl")
            except Exception:
                print("SETTLED.xlsx vazio ou inválido. Criando histórico novo.")
                hist = pd.DataFrame(columns=novo.columns)

        else:
            hist = pd.DataFrame(columns=novo.columns)

        hist.columns = hist.columns.astype(str).str.strip()

        # =====================================================
        # Garantir colunas essenciais
        # =====================================================

        for col in [col_pasta, col_status, col_data_cadastro]:

            if col not in novo.columns:
                raise ValueError(f"Coluna obrigatória '{col}' não encontrada em SETTLED_2.xlsx")

            if col not in hist.columns:
                hist[col] = None

        # =====================================================
        # Padronizar campos
        # =====================================================

        novo[col_pasta] = novo[col_pasta].astype(str).str.strip()
        hist[col_pasta] = hist[col_pasta].astype(str).str.strip()

        novo[col_data_cadastro] = pd.to_datetime(novo[col_data_cadastro], errors="coerce")
        hist[col_data_cadastro] = pd.to_datetime(hist[col_data_cadastro], errors="coerce")

        # =====================================================
        # Remover duplicados
        # =====================================================

        novo = novo.drop_duplicates(subset=[col_pasta], keep="last")
        hist = hist.drop_duplicates(subset=[col_pasta], keep="last")

        # =====================================================
        # Merge histórico + novo
        # =====================================================

        merged = hist.merge(
            novo,
            on=col_pasta,
            how="outer",
            suffixes=("_hist", "_novo")
        )

        # =====================================================
        # Escolher registro mais recente (corrigir perda de linhas)
        # =====================================================
        novo_existe = (
            merged[f"{col_status}_novo"].notna() |
            merged[f"{col_data_cadastro}_novo"].notna()
        )

        cond_atualizar = novo_existe & (
            (merged[f"{col_status}_hist"] != merged[f"{col_status}_novo"]) |
            (merged[f"{col_data_cadastro}_novo"] > merged[f"{col_data_cadastro}_hist"])
        )

        # linhas que vêm do novo
        atualizar = merged[cond_atualizar]

        # linhas que mantêm histórico
        manter = merged[~cond_atualizar]

        # =====================================================
        # Reconstruir dataframe final (CORRIGIDO: incluir "Pasta")
        # =====================================================

        df_manter = manter[[c for c in merged.columns if c.endswith("_hist")]].rename(
            columns=lambda x: x.replace("_hist", "")
        ).copy()

        df_atualizar = atualizar[[c for c in merged.columns if c.endswith("_novo")]].rename(
            columns=lambda x: x.replace("_novo", "")
        ).copy()

        # Garantir que "Pasta" está presente em ambos os dataframes
        if col_pasta not in df_manter.columns and col_pasta in manter.columns:
            df_manter[col_pasta] = manter[col_pasta]

        if col_pasta not in df_atualizar.columns and col_pasta in atualizar.columns:
            df_atualizar[col_pasta] = atualizar[col_pasta]

        df_atualizado = pd.concat([df_manter, df_atualizar], ignore_index=True)
        df_atualizado = df_atualizado.reset_index(drop=True)

        # =====================================================
        # Salvar de forma segura
        # =====================================================

        temp_file = "SETTLED_temp.xlsx"
        df_atualizado.to_excel(temp_file, index=False)

        if os.path.exists("SETTLED.xlsx"):
            os.remove("SETTLED.xlsx")

        os.rename(temp_file, "SETTLED.xlsx")

        print("ETAPA 11 concluída ✔")
        print("Total histórico:", len(df_atualizado))
        print(f"Coluna 'Pasta' presente: {col_pasta in df_atualizado.columns}")

ETAPA 11 concluída ✔
Total histórico: 34
Coluna 'Pasta' presente: True


ETAPA 12.1 — Filtro BacenJud

In [18]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_settled = "SETTLED.xlsx"
arquivo_bacenjud = "BACENJUD.xlsx"

if not os.path.exists(arquivo_settled):
    print("SETTLED.xlsx não existe. Filtro BacenJud ignorado.")

else:
    try:
        settled = pd.read_excel(arquivo_settled)
    except EmptyDataError:
        print("SETTLED.xlsx está vazio. Filtro BacenJud ignorado.")
        settled = pd.DataFrame()

    if settled.empty:
        print("SETTLED sem dados. Filtro BacenJud ignorado.")

    else:

        col_garantia = "Garantia"

        if col_garantia in settled.columns:

            bacenjud = settled[
                settled[col_garantia] == "03 - Penhora Online (BacenJud)"
            ].copy()

            # salvar nova planilha
            bacenjud.to_excel(arquivo_bacenjud, index=False)

            print("Planilha BACENJUD criada.")
            print("Total de casos BacenJud:", len(bacenjud))

        else:
            print("Coluna Garantia não encontrada. Filtro BacenJud ignorado.")

Planilha BACENJUD criada.
Total de casos BacenJud: 1


ETAPA 12.2 — Pagamentos (PROCV por Pasta)

In [19]:
import pandas as pd

arquivo_excel = "Pagamentos.xlsx"

pag = pd.read_excel(arquivo_excel)


In [20]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

if not os.path.exists("SETTLED.xlsx"):
    print("SETTLED.xlsx não existe. Pagamentos não aplicados.")

else:
    try:
        settled = pd.read_excel("SETTLED.xlsx", dtype=str)
    except EmptyDataError:
        print("SETTLED.csv está vazio. Pagamentos não aplicados.")
        settled = pd.DataFrame()

    if settled.empty:
        print("SETTLED sem dados. Pagamentos não aplicados.")

    else:
        if not os.path.exists("Pagamentos.xlsx"):
            print("Pagamentos.xlsx não encontrado.")
        
        else:
            pag = pd.read_excel("Pagamentos.xlsx", dtype=str)

            if pag.empty:
                print("Arquivo de pagamentos vazio. Nenhuma atualização feita.")

            else:
                # LIMPAR E NORMALIZAR NOMES DAS COLUNAS
                pag.columns = (
                    pag.columns
                    .str.replace(r"[\n\r]", " ", regex=True)
                    .str.replace('"', '', regex=False)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip()
                )

                settled.columns = (
                    settled.columns
                    .str.replace(r"[\n\r]", " ", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip()
                )

                print("=== COLUNAS ENCONTRADAS ===")
                print("Colunas em Pagamentos.xlsx:")
                print(pag.columns.tolist())
                print("\nColunas em SETTLED.xlsx:")
                print(settled.columns.tolist())
                print("===========================\n")

                col_pasta_settled = "Pasta"
                col_pasta_pag = "Nº Pasta Projuris"

                col_valor_lancamento = "Valor do Lançamento"
                col_venc_fatal = "VencFatal"
                col_venc_liquid = "VencLíquid"
                col_valor_integral = "Valor integral do Acordo/Condenação"

                # Verificar quais colunas existem
                colunas_pag_existem = {
                    col_pasta_pag: col_pasta_pag in pag.columns,
                    col_valor_lancamento: col_valor_lancamento in pag.columns,
                    col_venc_fatal: col_venc_fatal in pag.columns,
                    col_venc_liquid: col_venc_liquid in pag.columns,
                    col_valor_integral: col_valor_integral in pag.columns
                }

                print("Colunas esperadas em Pagamentos:")
                for col, existe in colunas_pag_existem.items():
                    print(f"  {col}: {'✓' if existe else '✗'}")
                print()

                if col_pasta_settled not in settled.columns:
                    print(f"❌ Coluna '{col_pasta_settled}' NÃO encontrada em SETTLED.xlsx")
                else:
                    print(f"✓ Coluna '{col_pasta_settled}' encontrada em SETTLED.xlsx")

                if col_pasta_pag not in pag.columns:
                    print(f"❌ Coluna '{col_pasta_pag}' NÃO encontrada em Pagamentos.xlsx")
                else:
                    print(f"✓ Coluna '{col_pasta_pag}' encontrada em Pagamentos.xlsx")

                # Se as colunas principais existem, fazer merge
                if col_pasta_settled in settled.columns and col_pasta_pag in pag.columns:

                    # Selecionar apenas colunas que existem
                    colunas_selecionaveis = [col_pasta_pag]
                    
                    if col_valor_lancamento in pag.columns:
                        colunas_selecionaveis.append(col_valor_lancamento)
                    if col_venc_fatal in pag.columns:
                        colunas_selecionaveis.append(col_venc_fatal)
                    if col_venc_liquid in pag.columns:
                        colunas_selecionaveis.append(col_venc_liquid)
                    if col_valor_integral in pag.columns:
                        colunas_selecionaveis.append(col_valor_integral)

                    pag_reduzido = pag[colunas_selecionaveis].copy()

                    # Renomear para manter compatibilidade
                    novos_nomes = {col_pasta_pag: col_pasta_settled}
                    pag_reduzido.rename(columns=novos_nomes, inplace=True)

                    # LEFT merge: mantém TODAS as linhas de SETTLED
                    # Pastas que não encontrarem match em Pagamentos ficarão com NaN
                    settled = settled.merge(
                        pag_reduzido,
                        on=col_pasta_settled,
                        how="left"
                    )

                    settled.to_excel(
                        "SETTLED.xlsx",
                        index=False,
                    )

                    print("\n✓ Pagamentos adicionados corretamente.")
                    print("✓ Pastas sem correspondência em Pagamentos mantidas com colunas vazias.")
                    print(f"Total de linhas em SETTLED: {len(settled)}")
                    print(f"Linhas com pagamentos: {settled[col_pasta_settled].notna().sum()}")

                else:
                    print("\n❌ Colunas principais não encontradas.")
                    print("Verifique os nomes das colunas nos arquivos.")


=== COLUNAS ENCONTRADAS ===
Colunas em Pagamentos.xlsx:
['Data de Lançamento', 'Unid de Negócio', 'Número do Processo', 'Advogado Erbe', 'Escritório', 'Empresa', 'Justificativa', 'Conta', 'Nº doc', 'Tipo', 'Nº Requisição', 'Nº Pasta Projuris', 'Nº Elaw', 'Instância', 'Descrição', 'Natureza', 'Nome da Parte', 'VencLíquid', 'VencFatal', 'Valor do Lançamento', 'Classificação da parcela', 'Tipo de pagamento', 'Valor integral do Acordo/Condenação', 'Pagamento editado', 'Análise FCX', 'Comprovante', 'OBSERVAÇÕES']

Colunas em SETTLED.xlsx:
['Numero atual do processo', 'Serial Import', 'Natureza', 'Autor', 'Réu', 'Status', 'Escritório', 'Natureza Financeira', 'Empreendimento', 'Unidade de Negócio', 'Resumo do processo', 'Assunto Principal', 'Jurisdição Atual', 'UF Foro', 'Procedimento', 'Fase Atual', 'Tipo de Ação', 'Motivo Encerramento', 'Data de Encerramento', 'Data de cadastro', 'Data de Início', 'Valor Pedido', 'Valor da causa', 'Advogado interno principal', 'Sentença', 'Previsão de Desem

ETAPA 13 — Tradução Macro Assunto 

In [21]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

# =========================
# mapa de classificação
# =========================

mapa_macro = {

    "FAR CEF - Vício Construtivo Área Comum": "Construction",
    "Reclamação Proprietário - Vício Construtivo Área Comum": "Construction",
    "Vício Construtivo - Área Comum": "Construction",
    "Atraso de Obra": "Delay",
    "Entrega do Imóvel": "Delay",
    "FAR - Vício Construtivo Unidade Autônoma": "FAR",
    "FAR CEF - Vício Construtivo Unidade Autônoma": "FAR",
    "Acidente de Trabalho": "Labor",
    "Dano Material l Moral": "Cível",
    "Esclarecimento de Informações": "Cível",
    "Horas Extras": "Labor",
    "Reclamação Trabalhista": "Labor",
    "Responsabilidade Solidária/Subsidiária": "Labor",
    "Alienação Fiduciária": "Cível",
    "Ambiental": "Tax",
    "Anulação de Cobrança": "Cível",
    "Arbitragem": "Cível",
    "Cobrança": "Tax",
    "Comissão de Corretagem": "Cível",
    "Corrupção": "Cível",
    "Cota Condominial": "Cível",
    "Desapropriação": "Cível",
    "Descumprimento da Oferta": "Cível",
    "DF Century - Desocupação": "Cível",
    "Embargo de Obra": "Cível",
    "Entrega de Documentos": "Cível",
    "Fornecedor": "Cível",
    "Hipoteca": "Cível",
    "Honorários Advocatícios": "Cível",
    "Incapacidade Financeira": "Cível",
    "Inexistência de Débito": "Cível",
    "Laudêmio": "Tax",
    "Leilão": "Cível",
    "Licenças Públicas": "Cível",
    "Multa Procon": "Tax",
    "Negativação Indevida": "Cível",
    "Obrigação de Fazer/Não Fazer": "Cível",
    "Outorga de Escritura": "Cível",
    "Parceria": "Cível",
    "Percentual Retido Distrato": "Cível",
    "Permutante": "Cível",
    "Posse": "Cível",
    "Protesto - Serviços (água, gás, energia, esgoto)": "Cível",
    "Relação de Consumo": "Cível",
    "Repasse": "Cível",
    "Rescisão Contratual": "Cível",
    "Rescisão Contratual - Locação": "Cível",
    "Restituição de Valores": "Cível",
    "Vizinho": "Cível",
    "Vício Construtivo - Unidade Autônoma": "Cível",
    "IPTU Cliente": "Property Tax",
    "Protesto - IPTU": "Property Tax",
    "CIDE": "Tax",
    "Contribuição Previdenciária": "Tax",
    "CREA": "Tax",
    "CSLL": "Tax",
    "Execução": "Tax",
    "ICMS": "Tax",
    "IE": "Tax",
    "IOF": "Tax",
    "IRPJ": "Tax",
    "ISS": "Tax",
    "ITBI": "Tax",
    "PerDcomp": "Tax",
    "PIS/COFINS": "Tax",
    "Preço Público": "Tax",
    "Protesto": "Tax",
    "Taxa": "Tax",
    "Taxa de Lixo": "Tax",
    "IPTU": "Tax",
    "Castelos D'água": "Cível"
}

# =========================
# arquivos para aplicar
# =========================

arquivos = [
    "SETTLED.xlsx",
    "Entradas_Pos_BP.xlsx",
    "Entradas_Analise.xlsx"
]

col_origem = "Assunto Principal"
col_destino = "Macro Assunto"

# =========================
# loop nos arquivos
# =========================

for arquivo in arquivos:

    if not os.path.exists(arquivo):
        print(f"{arquivo} não existe. Tradução ignorada.")
        continue

    try:
        df = pd.read_excel(arquivo)
    except EmptyDataError:
        print(f"{arquivo} está vazio. Tradução ignorada.")
        continue

    if df.empty:
        print(f"{arquivo} sem dados. Tradução ignorada.")
        continue

    df.columns = df.columns.str.strip()

    if col_origem not in df.columns:
        print(f"Coluna '{col_origem}' não encontrada em {arquivo}. Tradução ignorada.")
        continue

    # limpar assunto principal
    df[col_origem] = df[col_origem].astype(str).str.strip()

    # criar coluna Macro Assunto se não existir
    if col_destino not in df.columns:
        df[col_destino] = ""

    # identificar assuntos não classificados
    nao_classificados = df[df[col_origem].map(mapa_macro).isna()][col_origem].dropna().unique()

    if len(nao_classificados) > 0:
        print(f"\nAssuntos sem classificação em {arquivo}:")
        print(sorted(nao_classificados))

    # preencher macro assunto apenas se estiver vazio
    df[col_destino] = df[col_destino].fillna("").astype(str)

    mask = df[col_destino].str.strip() == ""

    df.loc[mask, col_destino] = df.loc[mask, col_origem].map(mapa_macro)

    # salvar
    df.to_excel(arquivo, index=False)

    print(f"Macro Assunto atualizado em {arquivo}")


Assuntos sem classificação em SETTLED.xlsx:
['nan']
Macro Assunto atualizado em SETTLED.xlsx

Assuntos sem classificação em Entradas_Pos_BP.xlsx:
['Descumprimento de NR', 'FAR - Vício Construtivo Área Comum', 'Foro', 'INSS']
Macro Assunto atualizado em Entradas_Pos_BP.xlsx

Assuntos sem classificação em Entradas_Analise.xlsx:
['Descumprimento de NR', 'Foro']
Macro Assunto atualizado em Entradas_Analise.xlsx


ETAPA 14 — MacroEncerramento

In [22]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

if not os.path.exists("SETTLED.xlsx"):
    print("SETTLED.xlsx não existe. Macro encerramento ignorado.")

else:
    try:
        df = pd.read_excel("SETTLED.xlsx")
    except EmptyDataError:
        print("SETTLED.xlsx está vazio. Macro encerramento ignorado.")
        df = pd.DataFrame()

    if df.empty:
        print("SETTLED sem dados. Macro encerramento ignorado.")

    else:
        col_macro = "Macro encerramento"
        col_motivo = "Motivo Encerramento"
        col_pasta = "Pasta"
        col_pagamento = "Valor integral do Acordo/Condenação"

        if col_motivo in df.columns and col_pasta in df.columns:

            # criar coluna se não existir
            if col_macro not in df.columns:
                df[col_macro] = ""

            mask_vazio = df[col_macro].isna() | (df[col_macro].astype(str).str.strip() == "")

            df_vazio = df[mask_vazio]

            df_vazio.to_excel("Relatorio_Macroencerramento.xlsx", index=False)

            valores_settled = [
                "Acordo",
                "Acordo pago - Aguardando baixa",
                "Acordo parcelado- Aguardando baixa",
                "REFIS/PPI - Aguardando baixa"
            ]

            mask_settled = (
                mask_vazio &
                df[col_motivo].isin(valores_settled) &
                ~df[col_motivo].str.contains("terceiro", case=False, na=False)
            )

            df.loc[mask_settled, col_macro] = "Settled"

            valores_lost = [
                "Condenação paga - Aguardando baixa",
                "Procedência"
            ]

            mask_lost = (
                mask_vazio &
                df[col_motivo].isin(valores_lost) &
                ~df[col_motivo].str.contains("terceiro", case=False, na=False)
            )

            df.loc[mask_lost, col_macro] = "Lost"

            if col_pagamento in df.columns:

                mask_restante = (
                    df[col_macro].isna() |
                    (df[col_macro].astype(str).str.strip() == "")
                )

                mask_won = (
                    mask_restante &
                    (df[col_pagamento].isna() |
                     (df[col_pagamento].astype(str).str.strip() == ""))
                )

                df.loc[mask_won, col_macro] = "Won"

            sobras = df[
                df[col_macro].isna() |
                (df[col_macro].astype(str).str.strip() == "")
            ]

            if len(sobras) > 0:
                print("Arquivos incorretos:")
                print(sobras[col_pasta].tolist())

            df.to_excel("SETTLED.xlsx", index=False)

            print("Macro encerramento atualizado.")

        else:
            print("Colunas necessárias não encontradas. Macro encerramento ignorado.")

Arquivos incorretos:
[36806.0, 36806.0, 36806.0, 36806.0, 36806.0, 36806.0, 36806.0, nan, nan]
Macro encerramento atualizado.


ETAPA FINAL — Consolidar Bases Históricas

In [23]:
import os
import pandas as pd

def consolidar_base(arquivo_origem, arquivo_destino):

    if not os.path.exists(arquivo_origem):
        print(f"{arquivo_origem} não encontrado.")
        return

    # Ler origem
    origem = pd.read_excel(arquivo_origem, engine="openpyxl")

    if origem.empty:
        print(f"{arquivo_origem} sem dados.")
        return

    # Se destino não existe → cria direto
    if not os.path.exists(arquivo_destino):
        origem.to_excel(arquivo_destino, index=False)
        print(f"{arquivo_destino} criado com {len(origem)} linhas.")
        return

    # Ler destino (uma única vez)
    try:
        destino = pd.read_excel(arquivo_destino, engine="openpyxl")
    except Exception:
        print(f"{arquivo_destino} corrompido. Recriando.")
        origem.to_excel(arquivo_destino, index=False)
        return

    # Concatenar direto (mais rápido)
    consolidado = pd.concat([destino, origem], ignore_index=True)

    consolidado.to_excel(arquivo_destino, index=False)

    print(f"{arquivo_destino} atualizado.")
    print(f"+{len(origem)} linhas adicionadas | Total: {len(consolidado)}")


# Executar
consolidar_base("ENTRADAS_ANALISE.xlsx", "Base_Entradas.xlsx")
consolidar_base("SETTLED.xlsx", "Base_Settled.xlsx")

print("\nConsolidação concluída ✔")

Base_Entradas.xlsx atualizado.
+268 linhas adicionadas | Total: 1299
Base_Settled.xlsx atualizado.
+67 linhas adicionadas | Total: 134

Consolidação concluída ✔


Painel Assumptions

In [26]:
import pandas as pd
import os

print("===== GERANDO ASSUMPTIONS_26 =====")

# ==============================
# VALORES FIXOS
# ==============================

# ALTERE AQUI caso os valores mudem no futuro
EXPECTED_LOSS_BASE = 597000
EXPECTED_LOSS_FIXO = 72600

# ALTERE AQUI caso o valor fixo do subtotal mude
SUBTOTAL_FIXO = 72600


# ==============================
# FUNÇÃO SEGURA PARA LIMPAR VALORES
# ==============================

def limpar_valor(col):

    # se já for número, retorna direto
    if pd.api.types.is_numeric_dtype(col):
        return pd.to_numeric(col, errors="coerce")

    return pd.to_numeric(
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip(),
        errors="coerce"
    )


# ==============================
# PROCESSOS SEM ATUALIZAÇÃO
# ==============================

sem_atualizacao = 0

if os.path.exists("relatorio_tratado.xlsx"):

    df = pd.read_excel("relatorio_tratado.xlsx")

    AZ = limpar_valor(df["Valor Pedido"])
    AY = limpar_valor(df["Valor Pedido Objeto Corrigido"])

    ativos = df["Status"].astype(str).str.upper() == "ATIVOS"

    regra = (
        ((AZ <= 1) & (AY <= 1)) |
        (abs(AZ - AY) < 0.01)
    )

    filtro = ativos & regra

    sem_atualizacao = AY[filtro].sum()

else:
    print("relatorio_tratado.xlsx não encontrado")


# ==============================
# PROCESSOS COM ATUALIZAÇÃO
# ==============================

com_atualizacao = 0

if os.path.exists("ATIVOS_COM_VARIACAO.xlsx"):

    ativos_var = pd.read_excel("ATIVOS_COM_VARIACAO.xlsx")

    AY = limpar_valor(ativos_var["Valor Pedido Objeto Corrigido"])

    com_atualizacao = AY.sum()

else:
    print("ATIVOS_COM_VARIACAO.xlsx não encontrado")


# ==============================
# SETTLED
# ==============================

finally_resolved = 0
savings = 0

if os.path.exists("Base_Settled.xlsx"):

    settled = pd.read_excel("Base_Settled.xlsx")

    AY = limpar_valor(settled["Valor Pedido Objeto Corrigido"])
    acordo = limpar_valor(settled["Valor integral do Acordo/Condenação"])

    finally_resolved = acordo.sum()

    savings = AY.sum() - acordo.sum()

else:
    print("Base_Settled.xlsx não encontrado")


# ==============================
# MUDANÇA DE PROGNÓSTICO
# ==============================

mudanca = 0

if os.path.exists("MUDANCA_PROGNOSTICO.xlsx"):

    mud = pd.read_excel("MUDANCA_PROGNOSTICO.xlsx")

    AY = limpar_valor(mud["Valor Pedido Objeto Corrigido"])

    mudanca = AY.sum()

else:
    print("MUDANCA_PROGNOSTICO.xlsx não encontrado")


# ==============================
# NEW CLAIMS
# ==============================

new_claims = 0

if os.path.exists("Entradas_Pos_BP.xlsx"):

    entradas = pd.read_excel("Entradas_Pos_BP.xlsx")

    AZ = limpar_valor(entradas["Valor Pedido"])

    ativos = entradas["Status"].astype(str).str.upper() == "ATIVOS"

    new_claims = AZ[ativos].sum()

else:
    print("Entradas_Pos_BP.xlsx não encontrado")


# ==============================
# EXPECTED LOSSES
# ==============================

expected_losses_col1 = EXPECTED_LOSS_BASE
expected_losses_col2 = EXPECTED_LOSS_FIXO
expected_losses_col3 = expected_losses_col1 + expected_losses_col2


# ==============================
# SUBTOTAL
# ==============================

subtotal_col1 = (
    expected_losses_col1
    + mudanca
    - finally_resolved
    - savings
)

subtotal_col2 = SUBTOTAL_FIXO
subtotal_col3 = subtotal_col1 + subtotal_col2


# ==============================
# EXPECTED LOSSES MENSAL
# ==============================

expected_losses_mensal = subtotal_col1 + new_claims


# ==============================
# DATAFRAME FINAL
# ==============================

assumptions = pd.DataFrame({

    "Descrição": [
        "Processos sem atualização financeira",
        "Processos com atualização financeira ativos",
        "Expected Losses",
        "Finally resolved claims",
        "Savings from finally resolved claims",
        "Changes in expected losses",
        "Subtotal",
        "New Claims",
        "Expected Losses (Mensal)"
    ],

    "Calculo": [
        sem_atualizacao,
        com_atualizacao,
        expected_losses_col1,
        finally_resolved,
        savings,
        mudanca,
        subtotal_col1,
        new_claims,
        expected_losses_mensal
    ],

    "Fixo": [
        "",
        "",
        expected_losses_col2,
        "",
        "",
        "",
        subtotal_col2,
        "",
        ""
    ],

    "Soma": [
        "",
        "",
        expected_losses_col3,
        "",
        "",
        "",
        subtotal_col3,
        "",
        ""
    ]
})


# ==============================
# AJUSTE DE ESCALA (DIVIDIR POR 1000)
# ==============================

linhas_dividir = [
    "Processos sem atualização financeira",
    "Processos com atualização financeira ativos",
    "Finally resolved claims",
    "Savings from finally resolved claims",
    "Changes in expected losses",
    "New Claims",
    "Expected Losses (Mensal)"
]

mask = assumptions["Descrição"].isin(linhas_dividir)

assumptions.loc[mask, "Calculo"] = pd.to_numeric(
    assumptions.loc[mask, "Calculo"],
    errors="coerce"
) / 1000


# ==============================
# SALVAR EXCEL
# ==============================

assumptions.to_excel("assumptions_26.xlsx", index=False)

print("assumptions_26.xlsx criado com sucesso")

===== GERANDO ASSUMPTIONS_26 =====
assumptions_26.xlsx criado com sucesso
